In [6]:
import sys 
print(sys.version)

3.14.6 | packaged by Anaconda, Inc. | (main, Jul  9 2026, 14:29:05) [MSC v.1942 64 bit (AMD64)]


In [13]:
import pandas as pd

In [14]:
import numpy as np

In [15]:
import matplotlib.pyplot as plt

In [16]:
.42print("Ok")

Ok


In [37]:
transactions = pd.DataFrame({
    "date": pd.to_datetime(["2026-01-05","2026-01-12","2026-02-02","2026-02-20",
                            "2026-03-10","2026-03-25","2026-04-08","2026-05-04"]),
    "ticker": ["AAPL","MSFT","AAPL","TSLA","MSFT","AAPL","TSLA","NVDA"],
    "sens":   ["BUY","BUY","BUY","BUY","SELL","SELL","BUY","BUY"],
    "quantite":[10,5,5,8,3,6,4,10],
    "prix":   [267.34,476.74,269.91,421.83,405.5,252.45,343.04,198.42], #en dollar
    "frais":  [1,1,1,1,1,1,1,1]
}).sort_values("date").reset_index(drop=True)
transactions

,date,ticker,sens,quantite,prix,frais
0,2026-01-05,AAPL,BUY,10,267.34,1
1,2026-01-12,MSFT,BUY,5,476.74,1
2,2026-02-02,AAPL,BUY,5,269.91,1
3,2026-02-20,TSLA,BUY,8,421.83,1
4,2026-03-10,MSFT,SELL,3,405.50,1
5,2026-03-25,AAPL,SELL,6,252.45,1
6,2026-04-08,TSLA,BUY,4,343.04,1
7,2026-05-04,NVDA,BUY,10,198.42,1


In [38]:
etat = {}          
realise = {}       
for _, t in transactions.iterrows():
    s = etat.setdefault(t.ticker, {"qte":0, "cout":0.0})
    realise.setdefault(t.ticker, 0.0)
    if t.sens == "BUY":
        s["qte"]  += t.quantite
        s["cout"] += t.quantite*t.prix + t.frais
    else:
        pru = s["cout"]/s["qte"]
        realise[t.ticker] += t.quantite*(t.prix - pru) - t.frais
        s["cout"] -= pru*t.quantite
        s["qte"]  -= t.quantite

positions = pd.DataFrame(etat).T
positions["pru"] = (positions["cout"]/positions["qte"]).replace([np.inf], np.nan)
positions["pnl_realise"] = pd.Series(realise)
positions = positions.reset_index().rename(columns={"index":"ticker"})
positions

,ticker,qte,cout,pru,pnl_realise
0,AAPL,9.0,2414.97,268.330000,-96.28
1,MSFT,2.0,953.88,476.940000,-215.32
2,TSLA,12.0,4748.80,395.733333,0.00
3,NVDA,10.0,1985.20,198.520000,0.00


In [39]:
prix_marche = {"AAPL":348.91, "MSFT":530.14, "TSLA":403.3, "NVDA":213.6}

#prix de cloture des titres en cette date du 30Octobre 2026 en dollar

In [40]:
positions["prix"] = positions.ticker.map(prix_marche)
positions["valeur"] = positions.qte * positions.prix
positions["pnl_latent"] = positions.valeur - positions.cout
positions["perf_%"] = positions.pnl_latent / positions.cout
positions["poids_%"] = positions.valeur / positions.valeur.sum()
positions["pnl_total"] = positions.pnl_realise + positions.pnl_latent
positions.round(2)

,ticker,qte,cout,pru,pnl_realise,prix,valeur,pnl_latent,perf_%,poids_%,pnl_total
0,AAPL,9.0,2414.97,268.33,-96.28,348.91,3140.19,725.22,0.30,0.28,628.94
1,MSFT,2.0,953.88,476.94,-215.32,530.14,1060.28,106.40,0.11,0.09,-108.92
2,TSLA,12.0,4748.80,395.73,0.00,403.30,4839.60,90.80,0.02,0.43,90.80
3,NVDA,10.0,1985.20,198.52,0.00,213.60,2136.00,150.80,0.08,0.19,150.80


In [41]:
secteurs = {"AAPL":"Tech","MSFT":"Tech","NVDA":"Tech","TSLA":"Auto"}
positions["secteur"] = positions.ticker.map(secteurs)
positions.groupby("secteur")["valeur"].sum() / positions.valeur.sum()

secteur
Auto    0.433032
Tech    0.566968
Name: valeur, dtype: float64

In [42]:
valeur0 = positions.valeur.sum()
chocs = [-0.30,-0.20,-0.10,0,0.10,0.20]
scenarios = pd.DataFrame({
    "choc": chocs,
    "valeur_portefeuille": [valeur0*(1+c) for c in chocs]
})
scenarios["impact_€"] = scenarios.valeur_portefeuille - valeur0
scenarios

,choc,valeur_portefeuille,impact_€
0,-0.3,7823.249,-3352.821
1,-0.2,8940.856,-2235.214
2,-0.1,10058.463,-1117.607
3,0.0,11176.070,0.000
4,0.1,12293.677,1117.607
5,0.2,13411.284,2235.214


In [45]:
%pip install xlsxwriter

Note: you may need to restart the kernel to use updated packages.


In [48]:
from pathlib import Path
import pandas as pd

for nom in ["transactions", "positions", "scenarios"]:
    if nom not in globals():
        raise ValueError(
            f"Le tableau '{nom}' manque. "
            "Réexécute les étapes 1 à 4 avant cette cellule."
        )

positions_excel = positions[
    [
        "ticker",
        "qte",
        "pru",
        "prix",
        "cout",
        "valeur",
        "pnl_realise",
        "pnl_latent",
        "pnl_total",
        "poids_%",
        "secteur"
    ]
].copy()


scenarios_excel = scenarios.copy()
scenarios_excel.columns = [
    "choc",
    "valeur_portefeuille",
    "impact_USD"
]

fichier = Path("dashboard.xlsx")

with pd.ExcelWriter(
    fichier,
    engine="xlsxwriter",
    datetime_format="dd/mm/yyyy"
) as writer:

    transactions.to_excel(
        writer,
        sheet_name="Transactions",
        index=False
    )

    positions_excel.to_excel(
        writer,
        sheet_name="Positions",
        index=False
    )

    scenarios_excel.to_excel(
        writer,
        sheet_name="Scenarios",
        index=False
    )

    classeur = writer.book

    format_montant = classeur.add_format({
        "num_format": '#,##0.00 "USD"'
    })

    format_pourcentage = classeur.add_format({
        "num_format": "0.0%"
    })

    format_quantite = classeur.add_format({
        "num_format": "0.##"
    })

    format_positif = classeur.add_format({
        "font_color": "#006100",
        "bg_color": "#C6EFCE"
    })

    format_negatif = classeur.add_format({
        "font_color": "#9C0006",
        "bg_color": "#FFC7CE"
    })

    for nom_feuille, tableau in [
        ("Transactions", transactions),
        ("Positions", positions_excel),
        ("Scenarios", scenarios_excel)
    ]:
        feuille = writer.sheets[nom_feuille]

        feuille.set_column(0, len(tableau.columns) - 1, 20)
        feuille.freeze_panes(1, 0)

        feuille.autofilter(
            0, 0,
            len(tableau), len(tableau.columns) - 1
        )

    feuille_transactions = writer.sheets["Transactions"]
    feuille_transactions.set_column("E:F", 18, format_montant)

    feuille_positions = writer.sheets["Positions"]

    feuille_positions.set_column("A:A", 14)
    feuille_positions.set_column("B:B", 12, format_quantite)
    feuille_positions.set_column("C:I", 20, format_montant)
    feuille_positions.set_column("J:J", 16, format_pourcentage)
    feuille_positions.set_column("K:K", 18)

    nombre_positions = len(positions_excel)

    for colonne in ["G", "H", "I"]:

        plage = f"{colonne}2:{colonne}{nombre_positions + 1}"

        feuille_positions.conditional_format(
            plage,
            {
                "type": "cell",
                "criteria": ">",
                "value": 0,
                "format": format_positif
            }
        )

        feuille_positions.conditional_format(
            plage,
            {
                "type": "cell",
                "criteria": "<",
                "value": 0,
                "format": format_negatif
            }
        )

    graphique_positions = classeur.add_chart({
        "type": "pie"
    })

    graphique_positions.add_series({
        "name": "Répartition des titres",
        "categories": [
            "Positions", 1, 0, nombre_positions, 0
        ],
        "values": [
            "Positions", 1, 5, nombre_positions, 5
        ],
        "data_labels": {
            "percentage": True
        }
    })

    graphique_positions.set_title({
        "name": "Répartition de la valeur des titres"
    })

    feuille_positions.insert_chart(
        "M2", graphique_positions
    )

    feuille_scenarios = writer.sheets["Scenarios"]

    feuille_scenarios.set_column(
        "A:A", 16, format_pourcentage
    )

    feuille_scenarios.set_column(
        "B:C", 26, format_montant
    )

    nombre_scenarios = len(scenarios_excel)

    graphique_scenarios = classeur.add_chart({
        "type": "column"
    })

    graphique_scenarios.add_series({
        "name": "Impact en USD",
        "categories": [
            "Scenarios", 1, 0, nombre_scenarios, 0
        ],
        "values": [
            "Scenarios", 1, 2, nombre_scenarios, 2
        ]
    })

    graphique_scenarios.set_title({
        "name": "Impact des scénarios de marché"
    })

    graphique_scenarios.set_x_axis({
        "name": "Choc sur les prix",
        "num_format": "0%"
    })

    graphique_scenarios.set_y_axis({
        "name": "Variation de valeur en USD"
    })

    feuille_scenarios.insert_chart(
        "E2", graphique_scenarios
    )

print("Le fichier Excel a été créé.")
print("Il se trouve ici :", fichier.resolve())

Le fichier Excel a été créé.
Il se trouve ici : C:\Users\evanl\dashboard.xlsx
